# 003 · defaultdict, OrderedDict and ChainMap

> **Standard Library › Data Structures** · Python 3.14

Three specialised mappings from `collections` that each remove a common chunk of boilerplate: `defaultdict` creates missing values on demand, `OrderedDict` adds order-manipulation methods on top of `dict`, and `ChainMap` looks keys up through a stack of dictionaries — ideal for layered configuration.

**In this notebook you will learn:**

- `defaultdict` and its factory function, and the grouping idiom (vs `GroupBy` / `ToLookup`)
- the `defaultdict` gotcha: *reading* a missing key inserts it
- why `OrderedDict` is still useful now that `dict` is ordered: `move_to_end`, `popitem(last=False)`, order-sensitive equality
- a tiny LRU cache built on `OrderedDict`
- `ChainMap` for layered lookups (command line → environment → defaults)

## .NET mental model

| .NET | Python |
|---|---|
| `dict.TryGetValue(k, out var v) ? v : dict[k] = new()` | `defaultdict(factory)` |
| `items.GroupBy(f)` / `items.ToLookup(f)` | `defaultdict(list)` + loop |
| `CollectionsMarshal.GetValueRefOrAddDefault` | `defaultdict` / `dict.setdefault` |
| `OrderedDictionary` / `OrderedDictionary<TKey,TValue>` (.NET 9) | `dict` (ordered since 3.7) or `OrderedDict` |
| LRU cache (hand-written: `Dictionary` + `LinkedList`) | `OrderedDict` with `move_to_end` — or `functools.lru_cache` |
| `IConfiguration` with layered providers (`Microsoft.Extensions.Configuration`) | `ChainMap(cli, env, defaults)` |

All three are `dict`-like (`defaultdict` and `OrderedDict` actually subclass `dict`), so the usual dictionary API applies *(see Syntax › Collections › 003 · Dictionaries)*.

## `defaultdict`: values created on demand

`defaultdict(factory)` behaves like a dict, except that accessing a missing key calls `factory()` (with no arguments), stores the result under that key, and returns it. The factory is usually a type: `int` (→ `0`), `list` (→ `[]`), `set` (→ `set()`).

```csharp
// C#
var index = new Dictionary<string, List<int>>();
if (!index.TryGetValue(key, out var list))
    index[key] = list = new List<int>();
list.Add(value);
```

In [ ]:
from collections import defaultdict

index = defaultdict(list)
index["fruit"].append("apple")       # "fruit" missing -> list() created, then appended
index["fruit"].append("pear")
index["veg"].append("carrot")
print(index)
print(dict(index))                   # convert to a plain dict for display / JSON

In [ ]:
counts = defaultdict(int)            # int() == 0
for ch in "banana":
    counts[ch] += 1
print(dict(counts))                  # (for counting, prefer collections.Counter)

## The grouping idiom

`defaultdict(list)` is the standard way to group items — the imperative equivalent of `GroupBy` / `ToLookup`. Unlike `itertools.groupby`, the input does **not** need to be sorted.

```csharp
// C#
var byDept = employees.ToLookup(e => e.Dept, e => e.Name);
```

In [ ]:
employees = [
    ("alice", "engineering"),
    ("bob", "sales"),
    ("carol", "engineering"),
    ("dave", "support"),
    ("erin", "sales"),
]

by_dept = defaultdict(list)
for name, dept in employees:
    by_dept[dept].append(name)

for dept, names in sorted(by_dept.items()):
    print(f"{dept:12} {names}")

Any zero-argument callable works as a factory — `set` for unique values, a `lambda` for a custom default, or even another `defaultdict` for nested structures:

In [ ]:
tags = defaultdict(set)
for post, tag in [("p1", "python"), ("p1", "tips"), ("p2", "python"), ("p1", "python")]:
    tags[post].add(tag)
print({k: sorted(v) for k, v in tags.items()})

prices = defaultdict(lambda: "n/a")
prices["apple"] = 1.2
print(prices["apple"], prices["kiwi"])

sales = defaultdict(lambda: defaultdict(int))     # region -> product -> units
sales["north"]["widgets"] += 5
sales["south"]["gadgets"] += 2
print({region: dict(p) for region, p in sales.items()})

⚠️ **Gotcha:** *reading* a missing key with `d[key]` **inserts** it. A lookup you meant to be harmless — logging, an `if d[key]:` check — silently grows the dict. Use `key in d` or `d.get(key)`, which do **not** call the factory.

In [ ]:
scores = defaultdict(list)
scores["alice"].append(90)

if scores["bob"]:               # looks like a read…
    print("bob has scores")
print(dict(scores))             # …but "bob" was inserted with []

print(scores.get("carol"), "carol" in scores)   # no insertion
print(sorted(scores))

**Tip:** for a one-off default on a regular `dict`, `d.setdefault(key, []).append(x)` does the same job without changing the dict type. And remember that a missing-key `KeyError` is *still* what you want in many places — `defaultdict` hides typos. Convert to a plain `dict` before handing the data to other code.

## `OrderedDict`: still useful

Since Python 3.7 a plain `dict` keeps insertion order, so `OrderedDict` is no longer needed just for ordering. It still adds three things `dict` lacks:

1. `move_to_end(key, last=True)` — move a key to either end in O(1)
2. `popitem(last=False)` — pop from the **front** (FIFO); `dict.popitem()` only pops the last item
3. **order-sensitive equality** between two `OrderedDict`s

In [ ]:
from collections import OrderedDict

od = OrderedDict(a=1, b=2, c=3)
od.move_to_end("a")                 # to the back
print(list(od))
od.move_to_end("c", last=False)     # to the front
print(list(od))

print(od.popitem(last=False))       # pop the FIRST item
print(od.popitem())                 # pop the last item (like dict.popitem)
print(od)

In [ ]:
d1, d2 = {"x": 1, "y": 2}, {"y": 2, "x": 1}
print(d1 == d2)                                    # True  — dict ignores order
print(OrderedDict(d1) == OrderedDict(d2))          # False — order matters
print(OrderedDict(d1) == d2)                       # True  — mixed comparison ignores order

## Sketch: an LRU cache

`move_to_end` + `popitem(last=False)` are exactly the two operations an LRU (least-recently-used) cache needs — in C# you'd combine a `Dictionary<K, LinkedListNode<…>>` with a `LinkedList<…>` to get the same O(1) behaviour.

In [ ]:
class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data: OrderedDict[str, object] = OrderedDict()

    def get(self, key: str):
        if key not in self.data:
            return None
        self.data.move_to_end(key)          # mark as most recently used
        return self.data[key]

    def put(self, key: str, value) -> None:
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            evicted, _ = self.data.popitem(last=False)   # drop least recently used
            print("evicted", evicted)

cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
cache.get("a")          # "a" is now most recent
cache.put("c", 3)       # evicts "b"
print(list(cache.data))

**Tip:** to cache *function results*, don't write this yourself — use the `@functools.lru_cache(maxsize=...)` decorator *(see Standard Library › Functional Tools › 003 · functools)*.

## `ChainMap`: layered lookups

`ChainMap(m1, m2, m3, ...)` groups several mappings into a single **view**. Lookups search each mapping in order and return the first hit; nothing is copied. It's the stdlib's take on layered configuration — like `ConfigurationBuilder` where later providers override earlier ones (in `ChainMap` the **first** map wins).

```csharp
// C# — later sources override earlier ones
var config = new ConfigurationBuilder()
    .AddInMemoryCollection(defaults)
    .AddEnvironmentVariables()
    .AddCommandLine(args)
    .Build();
```

In [ ]:
from collections import ChainMap

defaults = {"host": "localhost", "port": 8080, "debug": False}
env = {"port": 9000}                   # e.g. parsed from environment variables
cli = {"debug": True}                  # e.g. parsed from command-line arguments

config = ChainMap(cli, env, defaults)  # first match wins
print(config["host"], config["port"], config["debug"])
print(dict(config))                    # flatten to a regular dict

Because it's a live view, changes to the underlying dicts are visible immediately. **Writes** (and deletes) always go to the **first** mapping only — the lower layers are never modified.

In [ ]:
defaults["timeout"] = 30               # visible through the chain
print(config["timeout"])

config["port"] = 1234                  # written into `cli`, not into `env`
print(cli, env)
print(config.maps)                     # the underlying list of mappings

`new_child()` pushes a new empty (or given) layer on top and returns a new `ChainMap`; `.parents` returns the chain without the first layer. That makes it a neat model for nested scopes — e.g. variables in an interpreter or template contexts.

In [ ]:
globals_ = ChainMap({"x": 1, "y": 2})
local = globals_.new_child({"x": 10})    # inner scope shadows x
print(local["x"], local["y"])
print(local.parents["x"])                # look past the innermost scope
print(len(local), sorted(local))         # distinct keys across all layers

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Auto-create missing values | `TryGetValue` + add | `defaultdict(list)` |
| Group items by key | `ToLookup(key)` / `GroupBy(key)` | `defaultdict(list)` + `append` |
| Check without inserting | `ContainsKey` | `key in dd` / `dd.get(key)` |
| One-off default on a dict | `CollectionsMarshal.GetValueRefOrAddDefault` | `d.setdefault(key, default)` |
| Move key to end / front | — | `od.move_to_end(k, last=...)` |
| Pop oldest entry | — | `od.popitem(last=False)` |
| Order-sensitive equality | `SequenceEqual` on pairs | `OrderedDict == OrderedDict` |
| LRU cache | `Dictionary` + `LinkedList` | `OrderedDict` or `functools.lru_cache` |
| Layered config | `ConfigurationBuilder` | `ChainMap(cli, env, defaults)` |
| Add a scope layer | — | `chain.new_child()` |

## Exercises

1. Group the words `["apple", "avocado", "banana", "blueberry", "cherry"]` by their first letter using `defaultdict`.
2. Build an "inverted index" mapping each word to the set of document ids it appears in, from `{1: "red fish", 2: "blue fish", 3: "red car"}`.
3. Create a `ChainMap` of `user_settings`, `team_settings` and `defaults` and print the effective `theme` and `language`. Then show that setting a value through the chain only changes `user_settings`.
4. Write `dedupe_keep_last(pairs)` that keeps the **last** value for each key but orders keys by when they were **last** seen, using `OrderedDict.move_to_end`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from collections import ChainMap, OrderedDict, defaultdict

# 1
groups = defaultdict(list)
for w in ["apple", "avocado", "banana", "blueberry", "cherry"]:
    groups[w[0]].append(w)
print(dict(groups))

# 2
docs = {1: "red fish", 2: "blue fish", 3: "red car"}
inverted = defaultdict(set)
for doc_id, text in docs.items():
    for word in text.split():
        inverted[word].add(doc_id)
print({w: sorted(ids) for w, ids in sorted(inverted.items())})

# 3
defaults = {"theme": "light", "language": "en", "font_size": 12}
team_settings = {"language": "de"}
user_settings = {"theme": "dark"}
settings = ChainMap(user_settings, team_settings, defaults)
print(settings["theme"], settings["language"])
settings["font_size"] = 14
print(user_settings, defaults["font_size"])

# 4
def dedupe_keep_last(pairs):
    od = OrderedDict()
    for k, v in pairs:
        od[k] = v
        od.move_to_end(k)
    return list(od.items())

print(dedupe_keep_last([("a", 1), ("b", 2), ("a", 3), ("c", 4)]))
```

</details>